<a href="https://colab.research.google.com/github/BenCheung1/Pytorch_Sandbox/blob/main/Pyt_ConvolutionalNN_MNIST_Pt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# CONVOLUTIONAL NEURAL NETWORK
# Import Neural Network
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

# Importing torchvision - datasets
# Convert MNIST Image Files into a Tensor of 4-Dimensions
# (# of images, Heights, Width, Color Channel)
transform = transforms.ToTensor()

# torchvisions - datasets has various image datasets.
# see https://www.tensorflow.org/datasets/catalog/mnist
# datasets.MNIST indicates to get the MNIST numbers image set.
# Train Data, MNIST from datasets
train_data = datasets.MNIST(root='/cnn_data', train=True, download =True, transform=transform)



In [25]:
# Test Data, already downloaded MNIST Data
# Train = False, This is test Data not Train data.
test_data = datasets.MNIST(root='/cnn_data', train=False, download =True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [26]:
# Create a batch size for images, e.g. 10
train_loader = DataLoader(train_data, batch_size=10, shuffle= True)
test_loader = DataLoader(test_data, batch_size=10, shuffle= False)
# Define our CNN Model
# Describe convolutional layer and what it's doint
# two convolutional layers
# Conv2d: Input size #images=1, # filters outputs feature maps=6,
#         Kernel size=3 3x3, Stride length=1 one@time.
conv1 = nn.Conv2d(1, 6, 3, 1)
conv2 = nn.Conv2d(6, 16, 3, 1)
# Grab 1 MNIST record.
for i, (X_Train, y_train) in enumerate(train_data):
  break

In [27]:
# One image, size of the image is 28x28 pixels
X_Train.shape
# OUTPUT: torch.Size ([1, 28, 28])

torch.Size([1, 28, 28])

In [28]:
# Change this to 4-dimensions
x = X_Train.view(1,1,28,28)
# Perform first convolution
# Rectified Linear Unit for Activation Function
x = F.relu(conv1(x))
x.shape
# OUTPUT: torch.Size([1,6,26,26])
# 1 is the # of images, 6 is the # of filters feature maps
# 26x26 is the image.

torch.Size([1, 6, 26, 26])

In [29]:
# Pass Through the Pooling Layer
x = F.max_pool2d(x,2,2)
# Kernal of 2 & Stride of 2.
x.shape
# Pooling a 26x26 down to 13x13 b/c of 2x2 Kernel w/ Stride of 2. 26/2=13

torch.Size([1, 6, 13, 13])

In [30]:
# Do the second convolutional layer
x = F.relu(conv2(x))

In [31]:
x.shape
# didn't set Padding, so lose 2 pixels around the outside.
# Pooling Layer
x = F.max_pool2d(x,2,2)

In [32]:
x.shape
# will become OUTPUT torch.Size 1,16,5,5 because 11/2 =5
# this rounds down because you can't invent data to round up
# at this point, a Conv-Layer1 > Pooling > Cov-Layer2 > Pooling > output

torch.Size([1, 16, 5, 5])

In [35]:
# Model Class
class ConvolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1,6,3,1) #input, output, kernel, stride
    self.conv2 = nn.Conv2d(6,16,3,1)
    # Fully Connected Layers
    self.fc1 = nn.Linear(5*5*16, 120)
    # after 2 Convo layers > 5x5. 120 Neurons
    self.fc2 = nn.Linear(120, 84)
    # Each Fully Connected Layer reduces # of neurons
    self.fc3 = nn.Linear(84, 10) # input, output
    # Forward Function
  def forward(self,X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X,2,2) # 2x2 kernal and stride 2
    # Second Pass
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X,2,2) # 2x2 kernal and stride 2
    # Flatten, Re-view to Flatten
    X = X.view (-1,16*5*5) # negative one vary the batch size.

    # Fully Connected Layers
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)
    return F.log_softmax(X, dim=1)

In [36]:
# Create an instance of the Model
torch.manual_seed(87)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [37]:
# Loss Function Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# Smaller the learning the rate, longer it will take to train
